# 프롬프트 엔지니어링을 위한 모범 사례

## 주요 용어

프롬프트 엔지니어링: AI 모델이 원하는 출력을 생성하도록 입력을 설계하고 수정하는 실습.

토큰화: 텍스트를 모델이 이해하고 처리할 수 있는 작은 단위인 토큰으로 변환하는 과정.

지시 조정된 LLM(Instruction-Tuned LLMs): 특정 지시를 통해 응답 정확성과 관련성을 개선하기 위해 세부 조정된 대규모 언어 모델(LLM).

출처: [Best practices for prompt engineering with the OpenAI API](https://help.openai.com/en/articles/6654000-best-practices-for-prompt-engineering-with-openai-api)

## 왜 프롬프트 엔지니어링이 필요한가?

_왜_ 프롬프트 엔지니어링이 필요한지 이야기해 봅시다.  
그 이유는 현재 LLM이 _신뢰할 수 있고 일관된 완성_ 을 달성하기 어렵게 만드는 여러 과제를 제시하기 때문입니다.  
프롬프트 작성 및 최적화에 노력을 기울이지 않으면 이러한 과제를 해결하기 어렵습니다. 

예를 들어:

1. **모델 응답은 확률적입니다.** _동일한 프롬프트_ 는 다른 모델이나 모델 버전에서 다른 응답을 생성할 가능성이 높습니다.  
그리고 _동일한 모델_ 에서도 다른 시간에 다른 결과를 생성할 수 있습니다.  
_프롬프트 엔지니어링 기술은 더 나은 가드레일을 제공하여 이러한 변동을 최소화하는 데 도움을 줄 수 있습니다_.
- 답변을 JSON으로만 출력해.
- 키 이름은 name, age, job만 사용해.
- 그 외 문장은 출력하지 마.

2. **모델은 응답을 조작할 수 있습니다.** 모델은 _크지만 유한한_ 데이터셋으로 사전 훈련되었기 때문에 해당 훈련 범위를 벗어난 개념에 대한 지식이 부족합니다.  
결과적으로 부정확하거나 상상적이거나 알려진 사실과 직접적으로 모순되는 완성을 생성할 수 있습니다.  
_프롬프트 엔지니어링 기술은 사용자가 AI에게 인용이나 추론을 요청하여 이러한 조작을 식별하고 완화하는 데 도움을 줄 수 있습니다_.
- 너가 확실하지 않은 내용은 "모르겠다"고 말해.
- 출력에는 반드시 출처나 이유를 포함해.
- 단계별로 생각하고 마지막에 근거를 써줘.


3. **모델의 능력은 다양합니다.** 최신 모델이나 모델 세대는 더 풍부한 기능을 제공하지만 비용 및 복잡성에서 고유한 특성과 트레이드오프를 가져옵니다.  
_프롬프트 엔지니어링은 모델별 요구 사항에 맞게 차이를 추상화하고 확장 가능하고 원활한 방식으로 적응할 수 있는 모범 사례와 워크플로를 개발하는 데 도움을 줄 수 있습니다_.
- 추론력 강화 → 단계적 사고 강제
- 지시 실행력 강화 → 출력 형식을 매우 상세히 정의
- 사실 기반 응답 강화 → 근거 제공 강제


OpenAI 또는 Azure OpenAI Playground에서 이를 직접 확인해 보세요:

- 동일한 프롬프트를 다른 LLM 배포(예: OpenAI, Azure OpenAI, Hugging Face)에서 사용해 보세요 - 변동을 확인했나요?
- 동일한 프롬프트를 _동일한_ LLM 배포(예: Azure OpenAI Playground)에서 반복적으로 사용해 보세요 - 이러한 변동은 어떻게 달랐나요?

In [16]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import AzureOpenAI

# 환경 변수 로드
dotenv_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=dotenv_path, override=True)

azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_openai_key = os.getenv("AZURE_OPENAI_KEY")
azure_openai_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2025-04-01-preview")
CHAT_COMPLETIONS_MODEL = os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME", "gpt-5.4-mini")

if not azure_openai_endpoint or not azure_openai_key:
    raise ValueError(".env 파일에 AZURE_OPENAI_ENDPOINT와 AZURE_OPENAI_KEY를 설정하세요.")

client = AzureOpenAI(
    azure_endpoint=azure_openai_endpoint,
    api_key=azure_openai_key,
    api_version=azure_openai_api_version,
    default_headers={"Ocp-Apim-Subscription-Key": azure_openai_key},
)

print(f"Azure OpenAI endpoint: {azure_openai_endpoint}")
print(f"API version: {azure_openai_api_version}")
print(f"Chat deployment: {CHAT_COMPLETIONS_MODEL}")

Azure OpenAI endpoint: https://apim-ai-workshop-010.azure-api.net/
API version: 2025-04-01-preview
Chat deployment: gpt-5.4-mini


# 1. 최신 모델 사용

최고의 결과를 위해 최신 모델을 사용하세요.

# 2. 프롬프트의 시작 부분에 지침을 배치하고 ### 또는 """로 지침과 컨텍스트를 구분하세요

In [17]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "당신은 도움이 되는 어시스턴트입니다."},
        {
            "role": "user",
            "content": """
아래 텍스트를 가장 중요한 요점의 글머리표 목록으로 한글로 요약해 주세요.

###
We’re happy to announce that OpenAI and Microsoft are extending our partnership.
This multi-year, multi-billion dollar investment from Microsoft follows their previous investments
in 2019 and 2021, and will allow us to continue our independent research and develop AI that is
increasingly safe, useful, and powerful.

In pursuit of our mission to ensure advanced AI benefits all of humanity, OpenAI remains a
capped-profit company and is governed by the OpenAI non-profit. This structure allows us to
raise the capital we need to fulfill our mission without sacrificing our core beliefs about
broadly sharing benefits and the need to prioritize safety.
Microsoft shares this vision and our values, and our partnership is instrumental to our progress.
###
""",
        },
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

- OpenAI와 Microsoft가 파트너십을 연장했다.
- Microsoft는 2019년과 2021년에 이어, 이번에도 수년간에 걸친 수십억 달러 규모의 투자를 진행한다.
- 이번 투자로 OpenAI는 독립적인 연구를 계속하고, 더 안전하고 유용하며 강력한 AI를 개발할 수 있게 된다.
- OpenAI는 “첨단 AI의 혜택이 인류 전체에 돌아가게 한다”는 사명을 추구한다.
- OpenAI는 여전히 수익 상한이 있는 회사이며, 비영리 조직인 OpenAI non-profit의 지배를 받는다.
- 이 구조는 필요한 자본을 확보하면서도, 혜택의 폭넓은 공유와 안전 우선 원칙을 지킬 수 있게 한다.
- Microsoft는 이러한 비전과 가치를 공유하며, 이번 협력은 OpenAI의 발전에 매우 중요하다.


# 3. 원하는 컨텍스트, 결과, 길이, 형식, 스타일 등에 대해 구체적이고 상세하게 작성하세요

In [18]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "경복궁을 위한 시 한 편 써줘"},
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

물빛보다 고요한 지붕 아래  
시간은 돌계단을 천천히 오르고  

연못은 하늘의 마음을 받아  
한 번도 무너지지 않은 얼굴로  
오래된 바람을 비춘다  

기와 끝에 맺힌 햇살 한 점이  
조선의 아침처럼 맑게 번질 때  
나는 문득 알게 된다  

사라진 것들이 사라진 채로  
오히려 더 깊이 남아  
돌과 나무와 공기 사이에서  
오늘도 한 나라의 숨결이 된다는 것을  

경복궁이여,  
너는 과거가 아니라  
지금도 계속 자라나는  
조용한 왕좌 같다  

사람들의 발걸음이 스쳐 지나가도  
네 품은 흔들리지 않고  
늦은 오후의 그림자까지  
정성스레 품어 안는다  

그래서 나는 떠날 때마다  
한 줌의 그리움을 가져가고  
돌아올 때마다  
처음 보는 풍경처럼  
네 앞에서 다시 고개를 숙인다


In [19]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "경복궁을 위한 시 한 편 써줘. 특히 봄에서 여름으로 넘어가는 계절의 아름다운 풍경을 충분히 묘사해줘.",
        },
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

경복궁

봄은 돌담 끝에  
마지막 분홍을 걸어 두고  
천천히 물러난다.

경회루 연못 위로  
연잎은 아직 종잇장처럼 여려  
바람 한 줄기에도  
초록의 숨을 고른다.

향원정 가는 길,  
매화의 흰 그림자는 옅어지고  
그 자리에  
젖은 잔디의 향기와  
젊은 잎사귀의 빛이 차오른다.

근정전 처마 끝  
금빛 단청은 아침 햇살을 받아  
더 또렷해지고,  
기와 위로 흘러내린 햇빛은  
마치 오래된 시간을  
따뜻하게 데우는 손길 같다.

봄비가 지나간 자리마다  
돌계단은 맑아지고  
회색의 궁궐은  
어느새 초록을 품은 채  
조용히 숨을 넓힌다.

이 계절의 경복궁은  
꽃보다 잎이 먼저 말을 걸고  
바람보다 그늘이 먼저 도착한다.  
튤립처럼 선명한 봄의 끝자락에서  
연초록 여름은  
나뭇가지 끝에 조용히 매달려  
곧 피어날 모든 것들을  
미리 축복한다.

나는 그 사이를 걷는다.  
돌과 물과 나무와 시간이  
한 겹씩 포개지는 길 위에서,  
경복궁은  
봄이 여름에게 건네는  
가장 우아한 인사처럼  
아름답게 서 있다.


# 4. 원하는 출력 형식을 예제를 통해 명확히 표현하세요 (예제 1, 예제 2).

In [20]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": """
아래 텍스트에서 회사명과 연도를 추출하고 각 엔티티의 시작 인덱스와 끝 인덱스를 출력합니다.
출력 예: {"text": "OpenAI", "start": 28, "end": 34}

###
OpenAI와 Microsoft가 파트너십을 연장한다는 기쁜 소식을 전하게 되어 기쁩니다.
Microsoft의 이번 다년간, 수십억 달러 규모의 투자는 2019년과 2021년에 이루어진 이전 투자에 이은 것입니다.
2019년과 2021년 투자에 이은 것으로, 이를 통해 우리는 독립적인 연구를 계속하고 더욱 안전하고 유용하며 강력한 AI를 개발할 수 있게 될 것입니다.
###
""",
        },
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

{"text": "OpenAI", "start": 0, "end": 6}
{"text": "Microsoft", "start": 7, "end": 16}
{"text": "Microsoft", "start": 43, "end": 52}
{"text": "2019년", "start": 68, "end": 73}
{"text": "2021년", "start": 76, "end": 81}
{"text": "2019년", "start": 87, "end": 92}
{"text": "2021년", "start": 95, "end": 100}


In [21]:
prompt = """
아래 텍스트에 언급된 중요한 엔티티를 추출합니다.
먼저 모든 회사 이름을 추출한 다음 모든 연도를 추출합니다.
그런 다음 콘텐츠에 맞는 특정 주제를 추출하고 마지막으로 일반적인 주요 주제를 추출합니다.

Desired format:
Company names: <comma_separated_list_of_company_names>
Years: -||-
Specific topics: -||-
General themes: -||-

###
OpenAI와 Microsoft가 파트너십을 연장한다는 기쁜 소식을 알려드리게 되어 기쁩니다.
Microsoft의 이번 다년간, 수십억 달러 규모의 투자는 2019년과 2021년 투자에 이어
독립적인 연구를 지속하고 더욱 안전하고 유용하며 강력한 AI를 개발할 수 있게 할 것입니다.
###
"""

response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

Company names: OpenAI, Microsoft
Years: 2019, 2021
Specific topics: 파트너십 연장, 다년간 투자, 수십억 달러 규모 투자, 독립적 연구 지속, 더 안전하고 유용하며 강력한 AI 개발
General themes: 인공지능 협력, 기술 투자, AI 연구 개발, 기업 파트너십


# 5. 제로샷(zero-shot)으로 시작하고, 이후 퓨샷(few-shot) 예제를 제공하세요. 둘 다 효과가 없으면 미세 조정을 검토하세요

## 제로샷(Zero-shot) → 퓨샷(Few-shot) → 미세 조정(Fine-tuning)

![Deploy](image/Deploy.png)

In [22]:
prompt = """
OpenAI와 Microsoft가 파트너십을 연장한다는 기쁜 소식을 알려드리게 되어 기쁩니다.
Microsoft의 이번 다년간, 수십억 달러 규모의 투자는 2019년과 2021년 투자에 이어
독립적인 연구를 지속하고 더욱 안전하고 유용하며 강력한 AI를 개발할 수 있게 할 것입니다.
"""

response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Extract keywords from the corresponding texts below."},
        {"role": "user", "content": prompt},
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

OpenAI, Microsoft, 파트너십 연장, 다년간 투자, 수십억 달러 규모, 독립적 연구, 안전한 AI, 유용한 AI, 강력한 AI, AI 개발


In [23]:
system_prompt = """
당신은 유용한 조수입니다. 아래 텍스트에서 키워드를 추출하세요.

텍스트: Stripe는 웹 개발자가 웹 사이트와 모바일 애플리케이션에 결제 처리를 통합하는 데 사용할 수 있는 API를 제공합니다.
키워드: Stripe, 결제 처리, API, 웹 개발자, 웹사이트, 모바일 애플리케이션
###
텍스트: OpenAI는 텍스트를 이해하고 생성하는 데 능숙한 최첨단 언어 모델을 학습시켰습니다. API를 통해 이러한 모델에 액세스할 수 있으며, 언어 처리와 관련된 다양한 작업을 해결하는 데 사용할 수 있습니다.
키워드: OpenAI, 언어 모델, 텍스트 처리, API
"""

user_prompt = """
텍스트: OpenAI와 Microsoft가 파트너십을 연장한다는 소식을 전하게 되어 기쁘게 생각합니다.
Microsoft의 이번 다년간, 수십억 달러 규모의 투자는 2019년과 2021년에 이루어진 이전 투자에 이은 것입니다.
이를 통해 우리는 독립적인 연구를 지속하고 더욱 안전하고 유용하며 강력한 AI를 개발할 수 있게 될 것입니다.
키워드:
"""

response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

키워드: OpenAI, Microsoft, 파트너십 연장, 다년간 투자, 수십억 달러 규모, 독립적인 연구, 안전한 AI, 유용한 AI, 강력한 AI


# 6. 모호하고 부정확한 설명을 줄이세요

In [24]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "새 제품에 대한 설명을 작성하세요. 이 제품은 차세대 카시트입니다. 이 제품에 대한 설명은 몇 문장으로만 짧게 작성하고 너무 길지 않아야 합니다.",
        },
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

차세대 카시트는 아이의 안전과 편안함을 한층 더 높인 스마트한 설계가 돋보이는 제품입니다. 간편한 장착 방식과 향상된 충격 보호 기능으로 부모의 걱정을 덜어주며, 장거리 이동에서도 편안한 착석감을 제공합니다. 세련된 디자인까지 갖춰 실용성과 스타일을 모두 만족시키는 카시트입니다.


In [25]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": "새 제품에 대한 설명을 작성하세요. 이 제품은 차세대 카시트입니다. 3~5문장으로 구성된 단락을 사용하여 이 제품을 설명하세요.",
        },
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

차세대 카시트는 아이의 안전과 편안함을 한 단계 높인 혁신적인 제품입니다. 충격 흡수 구조와 향상된 고정 시스템을 적용해 주행 중에도 안정적인 보호를 제공하며, 성장 단계에 맞춰 세밀하게 조절할 수 있어 오래 사용할 수 있습니다. 또한 통기성이 뛰어난 소재와 인체공학적 설계로 장시간 탑승에도 쾌적함을 유지해 줍니다. 세련된 디자인까지 더해져 안전성과 스타일을 모두 갖춘 스마트한 선택입니다.


# 7. 하지 말아야 할 것을 말하는 대신, 해야 할 것을 명확히 설명하세요

In [26]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": """
The following is a conversation between an Agent and a Customer.
DO NOT ASK USERNAME OR PASSWORD. DO NOT REPEAT.

Customer: I can’t log in to my account.
Agent:
""",
        },
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

Sorry you’re having trouble logging in. Try resetting your password first, and make sure you’re using the correct email or username. If it still doesn’t work, I can help troubleshoot common sign-in issues.


In [30]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": """
The following is a conversation between an Agent and a Customer.
The agent will attempt to diagnose the problem and suggest a solution, while refraining from asking any questions related to PII.
Instead of asking for PII, such as username or password, refer the user to the login troubleshooting FAQ in the help center.

Customer: I can’t log in to my account.
Agent:
""",
        },
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

Sorry you’re having trouble logging in. I can help with general troubleshooting, but I can’t ask for personal login details.

Please try these steps:
1. Make sure you’re using the correct email/username and password.
2. Check that Caps Lock is off and try typing the password manually.
3. Clear your browser cache/cookies or try a different browser/device.
4. If available, use the “Forgot password” option to reset your password.
5. Make sure your account isn’t locked due to too many failed attempts.

If you still can’t get in, please refer to the login troubleshooting FAQ in the help center for more detailed steps.


# 8. 코드 생성 관련 - 특정 패턴으로 모델을 유도하기 위해 "선도 단어"를 사용하세요

In [28]:
response = client.chat.completions.create(
    model=CHAT_COMPLETIONS_MODEL,
    messages=[
        {"role": "system", "content": "You write only Python code. Do not include explanations or Markdown fences."},
        {
            "role": "user",
            "content": """
다음 요구사항을 만족하는 간단한 파이썬 코드를 작성합니다.
1. 마일 단위 숫자를 입력받습니다.
2. 마일을 킬로미터로 변환합니다.
3. 결과를 출력합니다.

아래 선도 단어 다음부터 코드를 이어서 작성하세요.

import
""",
        },
    ],
    max_completion_tokens=400,
)

print(response.choices[0].message.content)

import sys

miles = float(input())
kilometers = miles * 1.60934
print(kilometers)


# 9. Generate Anything 기능으로 프롬프트 초안 만들기

OpenAI Help Center 문서에는 **Generate Anything** 기능도 프롬프트 작성 보조 방법으로 소개되어 있습니다. 개발자가 원하는 작업이나 기대하는 출력 형식을 자연어로 설명하면, 그 목적에 맞는 프롬프트 초안을 만드는 데 도움을 받을 수 있습니다.

워크숍에서는 직접 기능을 호출하지는 않지만, 실무에서는 다음과 같은 방식으로 활용할 수 있습니다.

- 원하는 작업을 자연어로 먼저 설명합니다.
- 모델이 생성한 프롬프트 초안을 검토합니다.
- 출력 형식, 예시, 제약 조건을 추가해 프롬프트를 다듬습니다.
- 실제 API 호출에서 결과가 안정적인지 테스트합니다.

참고: [Using Generate Anything](https://help.openai.com/en/articles/9824968)

---

## Parameters 관련 참고

업데이트된 문서에서는 자주 조정하는 파라미터로 `model`, `temperature`, `max_completion_tokens`, `stop`을 언급합니다.

- `model`: 더 최신이거나 성능이 높은 모델일수록 지시를 더 잘 따르는 경향이 있습니다.
- `temperature`: 창의성과 무작위성을 조정합니다. 사실 기반 Q&A나 추출 작업은 낮은 값이 더 적합합니다.
- `max_completion_tokens`: 답변을 자연스럽게 요약하는 기능이 아니라, 생성 가능한 출력 토큰의 **상한선**입니다.
- `stop`: 특정 문자열이 생성되면 응답을 멈추도록 하는 중지 시퀀스입니다.

이 노트북의 코드 예제는 현재 Azure OpenAI Chat Completions API에 맞춰 `max_completion_tokens`를 사용합니다.